# Wind Model Comparison Analysis

## Research Objective
Compare wind speed predictions from three ECMWF models (IFS, AIFS-Single, AIFS-Ensemble) against actual observations (analysis data) over the past 7 days to determine if models systematically underestimate wind speeds and identify trends in high wind events.

## Analysis Phases
1. **Setup & Configuration** - Import libraries and define parameters
2. **Data Acquisition** - Download forecast data from ECMWF
3. **Data Loading & Preprocessing** - Parse GRIB2 files
4. **Wind Speed Calculation** - Compute magnitudes from components
5. **Statistical Comparison** - Calculate bias, RMSE, correlation
6. **Bias Analysis** - Evaluate model underestimation
7. **High Wind Events** - Analyze extreme wind performance
8. **Visualizations** - Create comprehensive plots
9. **Summary Report** - Generate findings and recommendations

In [ ]:
## STEP 1: Setup - Import Libraries
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import gridspec
import seaborn as sns
from ecmwf.opendata import Client
import cfgrib

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ All libraries imported successfully")

In [ ]:
## STEP 2: Configure Analysis Parameters
# Time period for analysis
ANALYSIS_DAYS = 7  # Analyze last 7 days
START_DATE = datetime.now() - timedelta(days=ANALYSIS_DAYS)
END_DATE = datetime.now()

# Wind parameters to analyze
PARAM_SFC = ["10u", "10v"]  # 10-meter wind components
PARAM_100M = ["100u", "100v"]  # 100-meter wind components
PARAM_GUSTS = ["10fg"]  # 10-meter wind gusts

# Forecast lead times to compare (in hours)
LEAD_TIMES = [24, 48, 72, 96, 120, 144]  # 1-6 days ahead

# Define high wind speed thresholds
HIGH_WIND_THRESHOLD_10M = 12.5  # m/s (>= high wind)
GUST_THRESHOLD = 15.0  # m/s (>= strong gust)

print(f"✓ Analysis period: {START_DATE.strftime('%Y-%m-%d')} to {END_DATE.strftime('%Y-%m-%d')}")
print(f"✓ Analyzing {len(LEAD_TIMES)} forecast lead times: {LEAD_TIMES}")
print(f"✓ High wind threshold: {HIGH_WIND_THRESHOLD_10M} m/s")
print(f"✓ Gust threshold: {GUST_THRESHOLD} m/s")

In [ ]:
## STEP 3: Initialize ECMWF Open Data Clients
# Create separate clients for each model
client_ifs = Client(
    source="ecmwf",
    model="ifs",
    resol="0p25",  # 0.25 degree resolution
    preserve_request_order=False,
    infer_stream_keyword=True
)

client_aifs_single = Client(
    source="ecmwf",
    model="aifs-single",
    resol="0p25",
    preserve_request_order=False,
    infer_stream_keyword=True
)

try:
    client_aifs_ens = Client(
        source="ecmwf",
        model="aifs-ens",
        resol="0p25",
        preserve_request_order=False,
        infer_stream_keyword=True
    )
    aifs_ens_available = True
    print("✓ AIFS-Ensemble client initialized")
except:
    aifs_ens_available = False
    print("⚠ AIFS-Ensemble not available (will compare IFS vs AIFS-Single only)")

print("✓ IFS client initialized")
print("✓ AIFS-Single client initialized")

In [ ]:
## STEP 4: Download Forecast Data for 7 Days
"""
Download wind data from all 3 models for the last 7 days.
For each day, we download forecasts from the 00 UTC run.
"""

download_summary = {}

for day_offset in range(1, ANALYSIS_DAYS + 1):
    print(f"\n{'='*60}")
    print(f"Downloading data for Day {day_offset} (date -({day_offset}))")
    print(f"{'='*60}")
    
    # IFS Download
    print(f"  → Downloading IFS forecast...")
    try:
        ifs_result = client_ifs.retrieve(
            date=-day_offset,
            time=0,
            type="fc",
            step=LEAD_TIMES,
            param=PARAM_SFC + PARAM_GUSTS,
            target=f"../../data/forecasts/ifs/ifs_day{day_offset}.grib2"
        )
        download_summary[f"IFS_day{day_offset}"] = {
            "file": f"../../data/forecasts/ifs/ifs_day{day_offset}.grib2",
            "datetime": ifs_result.datetime,
            "status": "✓ Success"
        }
        print(f"    ✓ Downloaded {ifs_result.datetime}")
    except Exception as e:
        download_summary[f"IFS_day{day_offset}"] = {
            "status": f"✗ Failed: {str(e)[:50]}"
        }
        print(f"    ✗ Failed: {e}")
    
    # AIFS-Single Download
    print(f"  → Downloading AIFS-Single forecast...")
    try:
        aifs_result = client_aifs_single.retrieve(
            date=-day_offset,
            time=0,
            type="fc",
            step=LEAD_TIMES,
            param=PARAM_SFC,
            target=f"aifs_single_day{day_offset}.grib2"
        )
        download_summary[f"AIFS_day{day_offset}"] = {
            "file": f"aifs_single_day{day_offset}.grib2",
            "datetime": aifs_result.datetime,
            "status": "✓ Success"
        }
        print(f"    ✓ Downloaded {aifs_result.datetime}")
    except Exception as e:
        download_summary[f"AIFS_day{day_offset}"] = {
            "status": f"✗ Failed: {str(e)[:50]}"
        }
        print(f"    ✗ Failed: {e}")
    
    # AIFS-Ensemble Download (if available)
    if aifs_ens_available:
        print(f"  → Downloading AIFS-Ensemble forecast...")
        try:
            aifs_ens_result = client_aifs_ens.retrieve(
                date=-day_offset,
                time=0,
                type="fc",
                step=LEAD_TIMES,
                param=PARAM_SFC,
                target=f"aifs_ens_day{day_offset}.grib2"
            )
            download_summary[f"AIFS_ENS_day{day_offset}"] = {
                "file": f"aifs_ens_day{day_offset}.grib2",
                "datetime": aifs_ens_result.datetime,
                "status": "✓ Success"
            }
            print(f"    ✓ Downloaded {aifs_ens_result.datetime}")
        except Exception as e:
            download_summary[f"AIFS_ENS_day{day_offset}"] = {
                "status": f"✗ Failed: {str(e)[:50]}"
            }

# Download Analysis Data (for comparison/verification)
print(f"\n{'='*60}")
print(f"Downloading ANALYSIS/OBSERVATIONS data")
print(f"{'='*60}")
print(f"  → Downloading IFS Analysis fields (step 0 and short steps)...")
try:
    analysis_result = client_ifs.retrieve(
        date=-1,  # Most recent
        time=0,
        type="fc",
        step=[0, 6, 12, 18, 24],  # Including some short lead times
        param=PARAM_SFC,
        target="analysis_winds.grib2"
    )
    print(f"    ✓ Downloaded analysis from {analysis_result.datetime}")
except Exception as e:
    print(f"    ✗ Failed: {e}")

print(f"\n✓ Download phase complete!")
print(f"\nDownload Summary:")
for key, value in download_summary.items():
    print(f"  {key}: {value['status']}")

In [ ]:
## STEP 5: Load and Parse GRIB2 Files
"""
Load the downloaded GRIB2 files using xarray and cfgrib engine.
Extract wind components and calculate wind speeds.
"""

def load_wind_data(filename):
    """Load GRIB2 file and return xarray dataset with wind speed calculated"""
    try:
        ds = xr.open_dataset(filename, engine="cfgrib")
        return ds
    except Exception as e:
        print(f"Error loading {filename}: {e}")
        return None

def calculate_wind_speed(u, v):
    """Calculate wind speed magnitude from u and v components"""
    return np.sqrt(u**2 + v**2)

# Dictionary to store all loaded datasets
datasets = {
    "IFS": {},
    "AIFS_Single": {},
    "AIFS_Ens": {},
    "Analysis": None
}

# Load IFS data
print("Loading IFS forecasts...")
for day in range(1, ANALYSIS_DAYS + 1):
    filename = f"../../data/forecasts/ifs/ifs_day{day}.grib2"
    try:
        ds = load_wind_data(filename)
        if ds is not None:
            # Calculate wind speed at 10m
            ds['wind_speed_10m'] = calculate_wind_speed(ds['u10'], ds['v10'])
            datasets["IFS"][day] = ds
            print(f"  ✓ Day {day}: {filename}")
    except Exception as e:
        print(f"  ✗ Day {day}: {e}")

# Load AIFS-Single data
print("\nLoading AIFS-Single forecasts...")
for day in range(1, ANALYSIS_DAYS + 1):
    filename = f"../../data/forecasts/aifs_single/aifs_single_day{day}.grib2"
    try:
        ds = load_wind_data(filename)
        if ds is not None:
            ds['wind_speed_10m'] = calculate_wind_speed(ds['u10'], ds['v10'])
            datasets["AIFS_Single"][day] = ds
            print(f"  ✓ Day {day}: {filename}")
    except Exception as e:
        print(f"  ✗ Day {day}: {e}")

# Load AIFS-Ensemble data (if available)
if aifs_ens_available:
    print("\nLoading AIFS-Ensemble forecasts...")
    for day in range(1, ANALYSIS_DAYS + 1):
        filename = f"../../data/forecasts/aifs_ensemble/aifs_ens_day{day}.grib2"
        try:
            ds = load_wind_data(filename)
            if ds is not None:
                ds['wind_speed_10m'] = calculate_wind_speed(ds['u10'], ds['v10'])
                datasets["AIFS_Ens"][day] = ds
                print(f"  ✓ Day {day}: {filename}")
        except Exception as e:
            print(f"  ✗ Day {day}: {e}")

# Load Analysis data
print("\nLoading Analysis data...")
try:
    ds_analysis = load_wind_data("../../data/observations/analysis_winds.grib2")
    if ds_analysis is not None:
        ds_analysis['wind_speed_10m'] = calculate_wind_speed(ds_analysis['u10'], ds_analysis['v10'])
        datasets["Analysis"] = ds_analysis
        print(f"  ✓ Analysis data loaded")
except Exception as e:
    print(f"  ✗ Analysis data: {e}")

print(f"\n✓ Data loading complete!")
print(f"  IFS files loaded: {len(datasets['IFS'])}/{ANALYSIS_DAYS}")
print(f"  AIFS-Single files loaded: {len(datasets['AIFS_Single'])}/{ANALYSIS_DAYS}")
if aifs_ens_available:
    print(f"  AIFS-Ensemble files loaded: {len(datasets['AIFS_Ens'])}/{ANALYSIS_DAYS}")

In [ ]:
## STEP 6: Extract Global Mean Wind Speeds
"""
For each model and day, calculate the global mean wind speed at each lead time.
This allows us to compare model performance at different forecast horizons.
"""

comparison_data = {
    "lead_time_hours": [],
    "date_offset": [],
    "IFS": [],
    "AIFS_Single": [],
    "AIFS_Ens": [],
    "Analysis": []
}

print("Extracting global mean wind speeds...\n")

# Process IFS data
for day in sorted(datasets["IFS"].keys()):
    ds = datasets["IFS"][day]
    wind_speeds = ds['wind_speed_10m'].mean(dim=['latitude', 'longitude']).values
    lead_times = ds['step'].values / np.timedelta64(1, 'h')
    
    for lead_time, speed in zip(lead_times, wind_speeds):
        comparison_data["lead_time_hours"].append(int(lead_time))
        comparison_data["date_offset"].append(f"Day -{day}")
        comparison_data["IFS"].append(float(speed))
        comparison_data["AIFS_Single"].append(np.nan)
        comparison_data["AIFS_Ens"].append(np.nan)
        comparison_data["Analysis"].append(np.nan)

# Process AIFS-Single data
for day in sorted(datasets["AIFS_Single"].keys()):
    ds = datasets["AIFS_Single"][day]
    wind_speeds = ds['wind_speed_10m'].mean(dim=['latitude', 'longitude']).values
    lead_times = ds['step'].values / np.timedelta64(1, 'h')
    
    for lead_time, speed in zip(lead_times, wind_speeds):
        # Find matching record and update
        matching_idx = None
        for i, (lt, do) in enumerate(zip(comparison_data["lead_time_hours"], 
                                           comparison_data["date_offset"])):
            if lt == int(lead_time) and do == f"Day -{day}":
                matching_idx = i
                break
        
        if matching_idx is not None:
            comparison_data["AIFS_Single"][matching_idx] = float(speed)
        else:
            comparison_data["lead_time_hours"].append(int(lead_time))
            comparison_data["date_offset"].append(f"Day -{day}")
            comparison_data["IFS"].append(np.nan)
            comparison_data["AIFS_Single"].append(float(speed))
            comparison_data["AIFS_Ens"].append(np.nan)
            comparison_data["Analysis"].append(np.nan)

# Process AIFS-Ensemble data if available
if aifs_ens_available and len(datasets["AIFS_Ens"]) > 0:
    for day in sorted(datasets["AIFS_Ens"].keys()):
        ds = datasets["AIFS_Ens"][day]
        wind_speeds = ds['wind_speed_10m'].mean(dim=['latitude', 'longitude']).values
        lead_times = ds['step'].values / np.timedelta64(1, 'h')
        
        for lead_time, speed in zip(lead_times, wind_speeds):
            matching_idx = None
            for i, (lt, do) in enumerate(zip(comparison_data["lead_time_hours"], 
                                               comparison_data["date_offset"])):
                if lt == int(lead_time) and do == f"Day -{day}":
                    matching_idx = i
                    break
            
            if matching_idx is not None:
                comparison_data["AIFS_Ens"][matching_idx] = float(speed)

# Create DataFrame
df_comparison = pd.DataFrame(comparison_data)

print(f"✓ Extracted global mean wind speeds from {len(df_comparison)} forecast points")
print(f"\nData Summary:")
print(f"  IFS observations: {df_comparison['IFS'].notna().sum()}")
print(f"  AIFS-Single observations: {df_comparison['AIFS_Single'].notna().sum()}")
if aifs_ens_available:
    print(f"  AIFS-Ens observations: {df_comparison['AIFS_Ens'].notna().sum()}")
print(f"\nFirst few rows:")
print(df_comparison.head(10))

In [ ]:
## STEP 7: Extract Analysis Data for Verification
"""
The analysis (actual observations) needs to be treated separately.
We use analysis fields at different lead times as "truth" for validation.
"""

if datasets["Analysis"] is not None:
    ds_analysis = datasets["Analysis"]
    analysis_wind_speeds = ds_analysis['wind_speed_10m'].mean(dim=['latitude', 'longitude']).values
    analysis_lead_times = ds_analysis['step'].values / np.timedelta64(1, 'h')
    
    # Calculate mean analysis wind speed (can also look at individual time steps)
    mean_analysis_speed = float(np.nanmean(analysis_wind_speeds))
    
    print(f"Analysis/Observation Data:")
    print(f"  Mean global wind speed: {mean_analysis_speed:.3f} m/s")
    print(f"  Min: {np.nanmin(analysis_wind_speeds):.3f} m/s")
    print(f"  Max: {np.nanmax(analysis_wind_speeds):.3f} m/s")
    print(f"  Std Dev: {np.nanstd(analysis_wind_speeds):.3f} m/s")
    
    print(f"\nAnalysis wind speeds at each step:")
    for lt, ws in zip(analysis_lead_times, analysis_wind_speeds):
        print(f"    Step {int(lt):3d}h: {ws:.3f} m/s")
else:
    print("⚠ Analysis data not available - using model climatology for reference")
    mean_analysis_speed = None

In [ ]:
## STEP 8: Statistical Comparison - Calculate Error Metrics
"""
Compare models against analysis (observations).
Calculate:
  - Bias: Mean difference (Forecast - Analysis)
  - MAE: Mean Absolute Error
  - RMSE: Root Mean Square Error
  - Correlation: Spatial/temporal correlation
"""

# Calculate mean wind speeds by model
df_by_leadtime = df_comparison.groupby('lead_time_hours').agg({
    'IFS': ['mean', 'std'],
    'AIFS_Single': ['mean', 'std'],
    'AIFS_Ens': ['mean', 'std']
}).round(3)

print("="*70)
print("WIND SPEED STATISTICS BY FORECAST LEAD TIME")
print("="*70)
print(df_by_leadtime)

# Get analysis reference (use mean across all time steps)
if mean_analysis_speed is not None:
    analysis_ref = mean_analysis_speed
else:
    analysis_ref = df_comparison[['IFS', 'AIFS_Single']].mean().mean()

print(f"\n{'='*70}")
print(f"REFERENCE ANALYSIS WIND SPEED: {analysis_ref:.3f} m/s")
print(f"{'='*70}\n")

# Calculate bias for each model
print("BIAS ANALYSIS (Forecast - Analysis)")
print("-" * 70)

metrics = {}
for model in ['IFS', 'AIFS_Single', 'AIFS_Ens']:
    valid_data = df_comparison[model].dropna()
    if len(valid_data) > 0:
        bias = (valid_data - analysis_ref).mean()
        mae = np.abs(valid_data - analysis_ref).mean()
        rmse = np.sqrt(((valid_data - analysis_ref)**2).mean())
        correlation = valid_data.corr(pd.Series(analysis_ref, index=valid_data.index))
        
        # Percentage bias
        pct_bias = (bias / analysis_ref) * 100
        
        metrics[model] = {
            'mean': valid_data.mean(),
            'std': valid_data.std(),
            'bias': bias,
            'pct_bias': pct_bias,
            'mae': mae,
            'rmse': rmse,
            'correlation': correlation,
            'n_samples': len(valid_data)
        }
        
        print(f"\n{model}:")
        print(f"  Mean wind speed: {valid_data.mean():.3f} ± {valid_data.std():.3f} m/s")
        print(f"  Bias: {bias:+.3f} m/s ({pct_bias:+.1f}%)")
        if pct_bias < 0:
            print(f"    → {model} UNDERESTIMATES winds by {abs(pct_bias):.1f}%")
        else:
            print(f"    → {model} OVERESTIMATES winds by {abs(pct_bias):.1f}%")
        print(f"  Mean Absolute Error (MAE): {mae:.3f} m/s")
        print(f"  Root Mean Square Error (RMSE): {rmse:.3f} m/s")
        print(f"  Correlation with analysis: {correlation:.3f}")
        print(f"  Sample size: {len(valid_data)} forecasts")

# Create metrics DataFrame for visualization
df_metrics = pd.DataFrame(metrics).T
print(f"\n{'='*70}")
print("METRICS SUMMARY TABLE")
print(f"{'='*70}")
print(df_metrics.round(4))

In [ ]:
## STEP 9: Analyze Bias by Forecast Lead Time
"""
Determine if bias changes with forecast lead time.
This helps identify if model skill degrades over time.
"""

print("\n" + "="*70)
print("BIAS EVOLUTION WITH FORECAST LEAD TIME")
print("="*70)

bias_by_leadtime = {}

for model in ['IFS', 'AIFS_Single', 'AIFS_Ens']:
    bias_by_leadtime[model] = []
    
    for lead_time in sorted(df_comparison['lead_time_hours'].unique()):
        subset = df_comparison[df_comparison['lead_time_hours'] == lead_time]
        valid_data = subset[model].dropna()
        
        if len(valid_data) > 0:
            bias = (valid_data - analysis_ref).mean()
            pct_bias = (bias / analysis_ref) * 100
            bias_by_leadtime[model].append({
                'lead_time': lead_time,
                'bias': bias,
                'pct_bias': pct_bias,
                'n': len(valid_data)
            })

# Print results
for model, biases in bias_by_leadtime.items():
    if len(biases) > 0:
        print(f"\n{model}:")
        print(f"  Lead Time (h) | Bias (m/s) | Bias (%) | Samples")
        print(f"  {'-'*55}")
        for b in biases:
            print(f"  {b['lead_time']:13.0f} | {b['bias']:10.3f} | {b['pct_bias']:8.1f} | {b['n']:7d}")

# Create dataframe for visualization
df_bias_leadtime = []
for model, biases in bias_by_leadtime.items():
    for b in biases:
        df_bias_leadtime.append({
            'model': model,
            'lead_time': b['lead_time'],
            'bias': b['bias'],
            'pct_bias': b['pct_bias']
        })

df_bias_leadtime = pd.DataFrame(df_bias_leadtime)
print(f"\n✓ Bias analysis by lead time complete")

In [ ]:
## STEP 10: High Wind Event Analysis
"""
Identify high wind events (>12.5 m/s) and analyze model performance
at extreme speeds. This reveals if models underestimate peak winds.
"""

print("\n" + "="*70)
print("HIGH WIND EVENT ANALYSIS (Wind Speed > 12.5 m/s)")
print("="*70)

# Identify high wind events in observations
if datasets["Analysis"] is not None:
    ds_analysis = datasets["Analysis"]
    
    # Find grid points and times with high winds
    high_wind_mask = ds_analysis['wind_speed_10m'] > HIGH_WIND_THRESHOLD_10M
    high_wind_events = high_wind_mask.sum(dim=['latitude', 'longitude'])
    
    print(f"\nHigh Wind Events in Analysis:")
    print(f"  Threshold: {HIGH_WIND_THRESHOLD_10M} m/s")
    print(f"  Time steps with high winds: {(high_wind_events > 0).sum().values}")
    print(f"  Max grid points with high winds (single time step): {high_wind_events.max().values:.0f}")
    print(f"  Mean grid points with high winds: {high_wind_events.mean().values:.1f}")

# Calculate model performance in high wind situations
print(f"\n{'Lead Time':<15} {'IFS':<20} {'AIFS-S':<20} {'AIFS-E':<20}")
print(f"{'':<15} {'Bias (m/s)':<20} {'Bias (m/s)':<20} {'Bias (m/s)':<20}")
print("-"*75)

high_wind_analysis = {}

for lead_time in sorted(df_comparison['lead_time_hours'].unique()):
    subset = df_comparison[df_comparison['lead_time_hours'] == lead_time]
    
    row = f"{int(lead_time):3d}h"
    
    for model in ['IFS', 'AIFS_Single', 'AIFS_Ens']:
        valid_data = subset[model].dropna()
        
        if len(valid_data) > 0:
            # Calculate bias for high wind subset
            # Since we're using global means, we look at high mean speeds
            high_wind_subset = valid_data[valid_data > HIGH_WIND_THRESHOLD_10M]
            
            if len(high_wind_subset) > 0:
                bias = (high_wind_subset - analysis_ref).mean()
                high_wind_analysis[f"{model}_{lead_time}"] = {
                    'bias': bias,
                    'n': len(high_wind_subset)
                }
                row += f" {bias:+7.3f}         "
            else:
                row += f" {'—':<20}"
    
    print(row)

print(f"\n✓ High wind analysis complete")

In [ ]:
## STEP 11: Visualization 1 - Wind Speed Comparison by Lead Time
"""
Create a comprehensive plot showing how wind speeds evolve
for each forecast lead time, comparing all three models.
"""

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Time series of mean wind speeds
ax1 = axes[0, 0]
for model in ['IFS', 'AIFS_Single', 'AIFS_Ens']:
    valid = df_comparison[df_comparison[model].notna()].copy()
    if len(valid) > 0:
        means = valid.groupby('lead_time_hours')[model].mean()
        ax1.plot(means.index, means.values, 'o-', label=model, linewidth=2.5, markersize=8)

if analysis_ref is not None:
    ax1.axhline(analysis_ref, color='red', linestyle='--', linewidth=2.5, label='Analysis Reference', alpha=0.8)

ax1.set_xlabel('Forecast Lead Time (hours)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Global Mean Wind Speed (m/s)', fontsize=12, fontweight='bold')
ax1.set_title('Wind Speed Evolution by Forecast Lead Time', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11, loc='best')
ax1.grid(True, alpha=0.3)

# Plot 2: Bias by lead time
ax2 = axes[0, 1]
if len(df_bias_leadtime) > 0:
    for model in ['IFS', 'AIFS_Single', 'AIFS_Ens']:
        model_data = df_bias_leadtime[df_bias_leadtime['model'] == model]
        if len(model_data) > 0:
            ax2.plot(model_data['lead_time'], model_data['bias'], 'o-', label=model, linewidth=2.5, markersize=8)

ax2.axhline(0, color='black', linestyle='-', linewidth=1, alpha=0.5)
ax2.set_xlabel('Forecast Lead Time (hours)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Bias (m/s)', fontsize=12, fontweight='bold')
ax2.set_title('Bias Evolution: Positive = Overestimate, Negative = Underestimate', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11, loc='best')
ax2.grid(True, alpha=0.3)

# Plot 3: Scatter plot - IFS vs Analysis
ax3 = axes[1, 0]
ifs_data = df_comparison[df_comparison['IFS'].notna()][['IFS']].copy()
if len(ifs_data) > 0:
    ax3.scatter(ifs_data.index, ifs_data['IFS'], alpha=0.6, s=50, label='IFS', color='blue')
    if analysis_ref is not None:
        ax3.axhline(analysis_ref, color='red', linestyle='--', linewidth=2, label='Analysis', alpha=0.8)
    ax3.set_xlabel('Forecast Index', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Wind Speed (m/s)', fontsize=12, fontweight='bold')
    ax3.set_title('IFS Wind Speed Distribution', fontsize=13, fontweight='bold')
    ax3.legend(fontsize=11)
    ax3.grid(True, alpha=0.3)

# Plot 4: Scatter plot - AIFS-Single vs Analysis
ax4 = axes[1, 1]
aifs_data = df_comparison[df_comparison['AIFS_Single'].notna()][['AIFS_Single']].copy()
if len(aifs_data) > 0:
    ax4.scatter(aifs_data.index, aifs_data['AIFS_Single'], alpha=0.6, s=50, label='AIFS-Single', color='green')
    if analysis_ref is not None:
        ax4.axhline(analysis_ref, color='red', linestyle='--', linewidth=2, label='Analysis', alpha=0.8)
    ax4.set_xlabel('Forecast Index', fontsize=12, fontweight='bold')
    ax4.set_ylabel('Wind Speed (m/s)', fontsize=12, fontweight='bold')
    ax4.set_title('AIFS-Single Wind Speed Distribution', fontsize=13, fontweight='bold')
    ax4.legend(fontsize=11)
    ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('wind_comparison_analysis.png', dpi=150, bbox_inches='tight')
print("✓ Saved: wind_comparison_analysis.png")
plt.show()

In [ ]:
## STEP 12: Visualization 2 - Model Comparison Metrics
"""
Create comparison charts showing performance metrics for each model
"""

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Prepare data
models_with_data = [m for m in ['IFS', 'AIFS_Single', 'AIFS_Ens'] if m in metrics and metrics[m]['n_samples'] > 0]
colors = {'IFS': 'steelblue', 'AIFS_Single': 'seagreen', 'AIFS_Ens': 'coral'}

# Plot 1: Mean Wind Speed Comparison
ax1 = axes[0, 0]
means = [metrics[m]['mean'] for m in models_with_data]
stds = [metrics[m]['std'] for m in models_with_data]
bars = ax1.bar(models_with_data, means, yerr=stds, capsize=10, 
               color=[colors[m] for m in models_with_data], alpha=0.7, edgecolor='black', linewidth=1.5)
if analysis_ref is not None:
    ax1.axhline(analysis_ref, color='red', linestyle='--', linewidth=2.5, label=f'Analysis ({analysis_ref:.2f} m/s)')
ax1.set_ylabel('Wind Speed (m/s)', fontsize=12, fontweight='bold')
ax1.set_title('Mean Wind Speed ± Std Dev', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for i, (bar, mean) in enumerate(zip(bars, means)):
    ax1.text(bar.get_x() + bar.get_width()/2, mean + stds[i] + 0.2, f'{mean:.2f}', 
             ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 2: Bias Comparison
ax2 = axes[0, 1]
biases = [metrics[m]['bias'] for m in models_with_data]
pct_biases = [metrics[m]['pct_bias'] for m in models_with_data]
bars = ax2.bar(models_with_data, biases, color=[colors[m] for m in models_with_data], 
               alpha=0.7, edgecolor='black', linewidth=1.5)
ax2.axhline(0, color='black', linestyle='-', linewidth=1)
ax2.set_ylabel('Bias (m/s)', fontsize=12, fontweight='bold')
ax2.set_title('Bias: Negative = Underestimate, Positive = Overestimate', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, bias, pct_bias in zip(bars, biases, pct_biases):
    ax2.text(bar.get_x() + bar.get_width()/2, bias + (0.05 if bias > 0 else -0.15), 
             f'{bias:.2f}\n({pct_bias:+.1f}%)', ha='center', va='bottom' if bias > 0 else 'top', 
             fontsize=10, fontweight='bold')

# Plot 3: RMSE Comparison
ax3 = axes[1, 0]
rmses = [metrics[m]['rmse'] for m in models_with_data]
bars = ax3.bar(models_with_data, rmses, color=[colors[m] for m in models_with_data], 
               alpha=0.7, edgecolor='black', linewidth=1.5)
ax3.set_ylabel('RMSE (m/s)', fontsize=12, fontweight='bold')
ax3.set_title('Root Mean Square Error (Lower is Better)', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, rmse in zip(bars, rmses):
    ax3.text(bar.get_x() + bar.get_width()/2, rmse + 0.05, f'{rmse:.3f}', 
             ha='center', va='bottom', fontsize=10, fontweight='bold')

# Plot 4: Correlation with Analysis
ax4 = axes[1, 1]
correlations = [metrics[m]['correlation'] if not np.isnan(metrics[m]['correlation']) else 0 
                for m in models_with_data]
bars = ax4.bar(models_with_data, correlations, color=[colors[m] for m in models_with_data], 
               alpha=0.7, edgecolor='black', linewidth=1.5)
ax4.set_ylabel('Correlation', fontsize=12, fontweight='bold')
ax4.set_ylim([0, 1.1])
ax4.set_title('Spatial/Temporal Correlation with Analysis (Higher is Better)', fontsize=13, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, corr in zip(bars, correlations):
    ax4.text(bar.get_x() + bar.get_width()/2, corr + 0.03, f'{corr:.3f}', 
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('model_metrics_comparison.png', dpi=150, bbox_inches='tight')
print("✓ Saved: model_metrics_comparison.png")
plt.show()

In [ ]:
## STEP 13: Trend Analysis - Identify High Wind Events
"""
Extract high wind events from forecast data and analyze model performance
specifically during periods of elevated wind speeds
"""

print("\n" + "="*70)
print("HIGH WIND TREND ANALYSIS")
print("="*70)

# Create wind speed categories
def categorize_wind_speed(speed):
    if speed < 5:
        return 'Light (< 5)'
    elif speed < 10:
        return 'Moderate (5-10)'
    elif speed < 12.5:
        return 'Fresh (10-12.5)'
    else:
        return 'Strong (≥12.5)'

# Analyze by wind speed category
category_analysis = {}

for model in ['IFS', 'AIFS_Single', 'AIFS_Ens']:
    valid_data = df_comparison[df_comparison[model].notna()].copy()
    
    if len(valid_data) > 0:
        valid_data['category'] = valid_data[model].apply(categorize_wind_speed)
        
        print(f"\n{model}:")
        for category in ['Light (< 5)', 'Moderate (5-10)', 'Fresh (10-12.5)', 'Strong (≥12.5)']:
            cat_data = valid_data[valid_data['category'] == category]
            if len(cat_data) > 0:
                mean_speed = cat_data[model].mean()
                count = len(cat_data)
                pct = (count / len(valid_data)) * 100
                print(f"  {category:<20}: {mean_speed:6.2f} m/s  ({count:3d} events, {pct:5.1f}%)")

# Identify forecasts with highest wind speeds
print(f"\n{'='*70}")
print("TOP 10 HIGHEST WIND SPEED FORECASTS")
print(f"{'='*70}")

top_winds = df_comparison[['lead_time_hours', 'date_offset', 'IFS', 'AIFS_Single', 'AIFS_Ens']].copy()
top_winds['max_wind'] = top_winds[['IFS', 'AIFS_Single', 'AIFS_Ens']].max(axis=1)
top_winds = top_winds.nlargest(10, 'max_wind')

print(top_winds.to_string())

print(f"\n✓ High wind trend analysis complete")

In [ ]:
## STEP 14: Visualization 3 - Distribution and Trend Analysis
"""
Visualize the distribution of wind speeds and bias patterns
"""

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Wind Speed Distribution (Histogram)
ax1 = axes[0, 0]
for model in ['IFS', 'AIFS_Single', 'AIFS_Ens']:
    valid = df_comparison[df_comparison[model].notna()][model]
    if len(valid) > 0:
        ax1.hist(valid, bins=20, alpha=0.5, label=model)
if analysis_ref is not None:
    ax1.axvline(analysis_ref, color='red', linestyle='--', linewidth=2.5, label='Analysis Mean')
ax1.set_xlabel('Wind Speed (m/s)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax1.set_title('Wind Speed Distribution', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# Plot 2: Error Distribution by Model
ax2 = axes[0, 1]
for model in ['IFS', 'AIFS_Single', 'AIFS_Ens']:
    valid = df_comparison[df_comparison[model].notna()][model]
    if len(valid) > 0:
        errors = valid - analysis_ref
        ax2.hist(errors, bins=20, alpha=0.5, label=model)
ax2.axvline(0, color='black', linestyle='-', linewidth=1.5)
ax2.set_xlabel('Forecast Error (m/s)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Frequency', fontsize=12, fontweight='bold')
ax2.set_title('Error Distribution (Forecast - Analysis)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

# Plot 3: Box Plot - Error by Lead Time
ax3 = axes[1, 0]
error_data_by_lead = []
lead_time_labels = []

for lead_time in sorted(df_comparison['lead_time_hours'].unique()):
    subset = df_comparison[df_comparison['lead_time_hours'] == lead_time]
    
    for model in ['IFS', 'AIFS_Single', 'AIFS_Ens']:
        valid = subset[model].dropna()
        if len(valid) > 0:
            errors = (valid - analysis_ref).values
            error_data_by_lead.append(errors)
            if model == 'IFS':
                lead_time_labels.append(f"{int(lead_time)}h")

if len(error_data_by_lead) > 0:
    bp = ax3.boxplot(error_data_by_lead, labels=lead_time_labels, patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
    ax3.axhline(0, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
    ax3.set_xlabel('Forecast Lead Time', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Error (m/s)', fontsize=12, fontweight='bold')
    ax3.set_title('Error Distribution by Forecast Lead Time', fontsize=13, fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='y')

# Plot 4: Time series of bias
ax4 = axes[1, 1]
for model in ['IFS', 'AIFS_Single', 'AIFS_Ens']:
    bias_data = df_comparison.copy()
    bias_data['bias'] = bias_data[model] - analysis_ref
    rolling_bias = bias_data.groupby('lead_time_hours')['bias'].mean()
    
    if len(rolling_bias) > 0:
        ax4.plot(rolling_bias.index, rolling_bias.values, 'o-', label=model, linewidth=2.5, markersize=8)

ax4.axhline(0, color='black', linestyle='-', linewidth=1)
ax4.set_xlabel('Forecast Lead Time (hours)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Mean Bias (m/s)', fontsize=12, fontweight='bold')
ax4.set_title('Bias Trend with Lead Time', fontsize=13, fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('distribution_and_trends.png', dpi=150, bbox_inches='tight')
print("✓ Saved: distribution_and_trends.png")
plt.show()

---

## Summary and Interpretation Guide

### Key Findings to Look For:

#### 1. **Systematic Bias**
   - **Negative bias** = Models underestimate wind speeds
   - **Positive bias** = Models overestimate wind speeds
   - If bias > -5%, models are reasonably accurate
   - If bias < -10%, models significantly underestimate

#### 2. **Model Comparison**
   - Which model (IFS vs AIFS-Single vs AIFS-Ens) is most accurate?
   - Do AI models (AIFS) perform better than traditional IFS?
   - Are differences statistically significant?

#### 3. **Lead Time Dependence**
   - Does bias increase with forecast lead time (skill degradation)?
   - Is one model more reliable at longer ranges?

#### 4. **High Wind Performance**
   - Do models underestimate during high wind events?
   - This is critical for wind energy and extreme weather prediction

#### 5. **Trends**
   - Identify days/periods with consistently high winds
   - Compare model behavior during these events
   - Look for systematic errors that correlate with wind regimes

In [ ]:
## STEP 15: Final Summary Report
"""
Generate a comprehensive summary of findings
"""

print("\n" + "="*80)
print("COMPREHENSIVE WIND MODEL ANALYSIS REPORT")
print("="*80)
print(f"\nAnalysis Period: {START_DATE.strftime('%Y-%m-%d')} to {END_DATE.strftime('%Y-%m-%d')} ({ANALYSIS_DAYS} days)")
print(f"Analysis Reference Wind Speed: {analysis_ref:.3f} m/s")

print("\n" + "-"*80)
print("1. MODEL ACCURACY & BIAS")
print("-"*80)

for model in ['IFS', 'AIFS_Single', 'AIFS_Ens']:
    if model in metrics and metrics[model]['n_samples'] > 0:
        m = metrics[model]
        print(f"\n{model}:")
        print(f"  • Mean forecast wind: {m['mean']:.3f} ± {m['std']:.3f} m/s")
        print(f"  • Bias: {m['bias']:+.3f} m/s ({m['pct_bias']:+.1f}%)")
        
        if m['pct_bias'] < -5:
            severity = "SIGNIFICANT UNDERESTIMATE"
        elif m['pct_bias'] < -2:
            severity = "Slight underestimate"
        elif m['pct_bias'] < 2:
            severity = "Unbiased / Accurate"
        elif m['pct_bias'] < 5:
            severity = "Slight overestimate"
        else:
            severity = "SIGNIFICANT OVERESTIMATE"
        
        print(f"  • Assessment: {severity}")
        print(f"  • RMSE: {m['rmse']:.3f} m/s")
        print(f"  • Correlation: {m['correlation']:.3f}")
        print(f"  • Sample size: {int(m['n_samples'])} forecasts")

print("\n" + "-"*80)
print("2. MODEL RANKING (Best to Worst)")
print("-"*80)

valid_metrics = {k: v for k, v in metrics.items() if v['n_samples'] > 0}
if valid_metrics:
    # Rank by absolute bias (closer to 0 is better)
    ranked = sorted(valid_metrics.items(), key=lambda x: abs(x[1]['pct_bias']))
    for i, (model, m) in enumerate(ranked, 1):
        print(f"{i}. {model:<20} (Bias: {m['pct_bias']:+6.1f}%, RMSE: {m['rmse']:.3f} m/s)")

print("\n" + "-"*80)
print("3. UNDERESTIMATION ANALYSIS")
print("-"*80)

for model in ['IFS', 'AIFS_Single', 'AIFS_Ens']:
    if model in metrics and metrics[model]['n_samples'] > 0:
        m = metrics[model]
        if m['pct_bias'] < 0:
            print(f"\n{model}: UNDERESTIMATES by {abs(m['pct_bias']):.1f}%")
            print(f"  → Wind speeds are forecasted {abs(m['bias']):.2f} m/s lower than observations")
            print(f"  → Impact: Potential safety risk for wind energy operations")
        elif m['pct_bias'] > 0:
            print(f"\n{model}: OVERESTIMATES by {m['pct_bias']:.1f}%")
            print(f"  → Wind speeds are forecasted {m['bias']:.2f} m/s higher than observations")
        else:
            print(f"\n{model}: UNBIASED")
            print(f"  → Wind speed forecasts are accurate on average")

print("\n" + "-"*80)
print("4. HIGH WIND PERFORMANCE")
print("-"*80)
print(f"High wind threshold: {HIGH_WIND_THRESHOLD_10M} m/s")
print(f"During high wind events, check the bias patterns above")
print("⚠ If models underestimate during HIGH winds, this is a critical issue")

print("\n" + "-"*80)
print("5. FORECAST LEAD TIME ANALYSIS")
print("-"*80)
if len(df_bias_leadtime) > 0:
    print("\nBias degradation with lead time:")
    for lead_time in sorted(df_comparison['lead_time_hours'].unique()):
        ifs_bias = df_bias_leadtime[(df_bias_leadtime['model'] == 'IFS') & 
                                    (df_bias_leadtime['lead_time'] == lead_time)]
        if len(ifs_bias) > 0:
            print(f"  {int(lead_time):3d}h lead: {float(ifs_bias['pct_bias']):+6.1f}% bias")

print("\n" + "-"*80)
print("6. RECOMMENDATIONS")
print("-"*80)

recos = []

for model in ['IFS', 'AIFS_Single', 'AIFS_Ens']:
    if model in metrics and metrics[model]['n_samples'] > 0:
        m = metrics[model]
        if m['pct_bias'] < -10:
            recos.append(f"• Apply correction factor of {abs(m['pct_bias']):.1f}% to {model} for operational use")
        if m['rmse'] > 3:
            recos.append(f"• {model} has high uncertainty (RMSE = {m['rmse']:.2f} m/s) - use ensemble approach")

if valid_metrics:
    best_model = ranked[0][0]
    recos.insert(0, f"• Use {best_model} as primary forecast model for accuracy")

if not recos:
    recos.append("• All models perform reasonably well within ±10% bias")
    recos.append("• Choose model based on lead time requirements")

for reco in recos:
    print(reco)

print("\n" + "="*80)
print("✓ ANALYSIS COMPLETE")
print("="*80)
print(f"\nGenerated visualizations:")
print("  1. wind_comparison_analysis.png - Time series and distributions")
print("  2. model_metrics_comparison.png - Performance metrics")
print("  3. distribution_and_trends.png - Statistical analysis")
print("\n✓ All results saved to workspace")